# Grafana + Prometheus Tutorial: Part 1

> Goal: verify the `grafana_tutorial` environment, load the Python monitoring stack, and start a tiny Flask app that exposes Prometheus metrics on `/metrics`.

> Before running the app cell, make sure this notebook is using the `Python (grafana_tutorial)` kernel in VS Code.

## What you will do

1. Confirm the notebook is running in the correct conda environment.
2. Import the packages used in this tutorial.
3. Start a small Flask service with Prometheus metrics.
4. Open the service locally and confirm Prometheus can scrape metrics.

## Expected result

- App endpoint: `http://127.0.0.1:8000/`
- Metrics endpoint: `http://127.0.0.1:8000/metrics`
- A counter named `tutorial_predictions_total` should appear in the metrics output.

In [31]:
from pathlib import Path
import sys
import importlib

project_dir = Path.cwd()
src_dir = project_dir / "src"
script_files = [
    src_dir / "app.py",
    src_dir / "concept_drift.py",
    src_dir / "data_drift.py",
    src_dir / "train.py",
]
required_packages = [
    "flask",
    "prometheus_client",
    "pandas",
    "numpy",
    "sklearn",
    "joblib",
    "requests",
    "apscheduler",
]

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Working directory:", project_dir)

if src_dir.exists():
    print(f"src directory already exists: {src_dir}")
else:
    src_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created src directory: {src_dir}")

for script_path in script_files:
    if script_path.exists():
        print(f"Script already exists: {script_path.name}")
    else:
        script_path.touch()
        print(f"Created script: {script_path.name}")

missing = []
for package_name in required_packages:
    try:
        importlib.import_module(package_name)
        print(f"OK: {package_name}")
    except Exception as exc:
        missing.append((package_name, str(exc)))
        print(f"MISSING: {package_name} -> {exc}")

if missing:
    raise RuntimeError(f"Missing packages: {missing}")

print("Environment check passed.")

Python executable: c:\Users\PROMET02\anaconda3\envs\grafana_tutorial\python.exe
Python version: 3.8.20
Working directory: d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring
src directory already exists: d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src
Script already exists: app.py
Script already exists: concept_drift.py
Script already exists: data_drift.py
Script already exists: train.py
OK: flask
OK: prometheus_client
OK: pandas
OK: numpy
OK: sklearn
OK: joblib
OK: requests
OK: apscheduler
Environment check passed.


## Step 2: Write training code

This section adds the training step that the tutorial needs before serving predictions.

The code cell below does four things:

- creates a `models` directory if needed
- writes a reusable training script to `src/train.py`
- trains a simple binary classifier on synthetic data
- saves the fitted model artifact to disk

That gives the later Flask app a concrete model artifact to load instead of relying on random-only behavior.

In [26]:
from pathlib import Path
import textwrap
import joblib
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

project_dir = Path.cwd()
src_dir = project_dir / "src"
models_dir = project_dir / "models"
train_script_path = src_dir / "train.py"
data_drift_path = src_dir / "data_drift.py"
concept_drift_path = src_dir / "concept_drift.py"
model_path = models_dir / "tutorial_model.joblib"
reference_data_path = models_dir / "reference_data.joblib"
feature_columns = [f"feature_{index}" for index in range(6)]

models_dir.mkdir(parents=True, exist_ok=True)

train_script = textwrap.dedent(
    '''
    from pathlib import Path

    import joblib
    import pandas as pd
    from sklearn.datasets import make_classification
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split



    FEATURE_COLUMNS = [f"feature_{index}" for index in range(6)]



    def build_reference_dataframe(X):
        return pd.DataFrame(X, columns=FEATURE_COLUMNS)



    def train_and_save_model(model_path: Path, reference_data_path: Path):
        X, y = make_classification(
            n_samples=1000,
            n_features=6,
            n_informative=4,
            n_redundant=0,
            random_state=42,
            class_sep=1.2,
        )
        X_df = build_reference_dataframe(X)
        X_train, X_test, y_train, y_test = train_test_split(
            X_df, y, test_size=0.2, random_state=42
        )
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        reference_data_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_path)
        joblib.dump(X_train.reset_index(drop=True), reference_data_path)
        return {
            "accuracy": score,
            "model_path": str(model_path),
            "reference_data_path": str(reference_data_path),
        }



    if __name__ == "__main__":
        models_dir = Path.cwd() / "models"
        result = train_and_save_model(
            models_dir / "tutorial_model.joblib",
            models_dir / "reference_data.joblib",
        )
        print(result)
    '''
).strip() + "\n"

data_drift_script = textwrap.dedent(
    '''
    import pandas as pd



    def detect_data_drift(reference_data: pd.DataFrame, current_data: pd.DataFrame, threshold: float = 0.12):
        drift_scores = {}
        for column in reference_data.columns:
            reference_mean = reference_data[column].mean()
            current_mean = current_data[column].mean()
            reference_std = reference_data[column].std() or 1.0
            drift_scores[column] = abs(current_mean - reference_mean) / reference_std

        overall_drift_score = sum(drift_scores.values()) / len(drift_scores)
        is_drift = overall_drift_score > threshold
        return is_drift, drift_scores, overall_drift_score
    '''
).strip() + "\n"

concept_drift_script = textwrap.dedent(
    '''
    from sklearn.metrics import accuracy_score



    def detect_concept_drift(
        model,
        X_reference,
        y_reference,
        X_current,
        y_current,
        threshold: float = 0.10,
    ):
        reference_predictions = model.predict(X_reference)
        current_predictions = model.predict(X_current)
        reference_accuracy = accuracy_score(y_reference, reference_predictions)
        current_accuracy = accuracy_score(y_current, current_predictions)
        relative_performance_drop = max(reference_accuracy - current_accuracy, 0.0)
        is_drift = relative_performance_drop > threshold
        return is_drift, relative_performance_drop
    '''
).strip() + "\n"

train_script_path.write_text(train_script, encoding="utf-8")
data_drift_path.write_text(data_drift_script, encoding="utf-8")
concept_drift_path.write_text(concept_drift_script, encoding="utf-8")
print(f"Wrote training script to {train_script_path}")
print(f"Wrote data drift helper to {data_drift_path}")
print(f"Wrote concept drift helper to {concept_drift_path}")

X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42,
    class_sep=1.2,
 )
X_df = pd.DataFrame(X, columns=feature_columns)
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
joblib.dump(model, model_path)
joblib.dump(X_train.reset_index(drop=True), reference_data_path)

print(f"Saved trained model to {model_path}")
print(f"Saved reference data to {reference_data_path}")
print(f"Validation accuracy: {accuracy:.3f}")

Wrote training script to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src\train.py
Wrote data drift helper to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src\data_drift.py
Wrote concept drift helper to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src\concept_drift.py
Saved trained model to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\models\tutorial_model.joblib
Saved reference data to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\models\reference_data.joblib
Validation accuracy: 0.725


## Step 3: Create a drift-aware monitored Flask app

This cell upgrades the toy service into a more realistic monitoring example that is closer to the DataCamp workflow.

The app now does five things:

- loads the trained model artifact from `models/tutorial_model.joblib`
- loads reference data from `models/reference_data.joblib`
- serves predictions from `/predict`
- updates Prometheus metrics for prediction activity, latency, data drift, and concept drift
- writes the same runnable application code to `src/app.py` so the notebook is not the only source of truth

That gives you a notebook-friendly Windows version of a model-monitoring service without requiring a separate Linux container build step.

In [ ]:
from pathlib import Path
from threading import Thread
from time import sleep
import sys

import joblib
import pandas as pd
from apscheduler.schedulers.background import BackgroundScheduler
from flask import Flask, jsonify, request
from prometheus_client import CollectorRegistry, Counter, Gauge, Histogram, generate_latest
from sklearn.datasets import make_classification
from werkzeug.serving import make_server

project_dir = Path.cwd()
src_dir = project_dir / "src"
models_dir = project_dir / "models"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from concept_drift import detect_concept_drift
from data_drift import detect_data_drift

model_path = models_dir / "tutorial_model.joblib"
reference_data_path = models_dir / "reference_data.joblib"
model = joblib.load(model_path)
reference_data = joblib.load(reference_data_path)
feature_columns = list(reference_data.columns)

app = Flask("grafana_prometheus_tutorial")
tutorial_registry = CollectorRegistry()

prediction_counter = Counter(
    "tutorial_predictions_total",
    "Total number of tutorial prediction requests",
    labelnames=["prediction_class", "status"],
    registry=tutorial_registry,
)
prediction_latency = Histogram(
    "tutorial_prediction_latency_seconds",
    "Latency of tutorial prediction requests",
    registry=tutorial_registry,
)
prediction_confidence_counter = Counter(
    "tutorial_prediction_confidence_total",
    "Predictions grouped by confidence band",
    labelnames=["confidence_band"],
    registry=tutorial_registry,
)
last_prediction = Gauge(
    "tutorial_last_prediction",
    "Latest prediction value returned by the tutorial app",
    registry=tutorial_registry,
)
last_prediction_confidence = Gauge(
    "tutorial_last_prediction_confidence",
    "Latest prediction confidence score returned by the tutorial app",
    registry=tutorial_registry,
)
data_drift_gauge = Gauge(
    "tutorial_data_drift_score",
    "Average feature drift score against the reference dataset",
    registry=tutorial_registry,
)
concept_drift_gauge = Gauge(
    "tutorial_concept_drift_score",
    "Relative performance drop between reference and current batches",
    registry=tutorial_registry,
)

def classify_prediction(score):
    return "positive" if score >= 0.5 else "negative"

def confidence_band(confidence):
    if confidence < 0.4:
        return "low"
    if confidence < 0.75:
        return "medium"
    return "high"

def build_current_batch(sample_size=200):
    X_current, y_current = make_classification(
        n_samples=sample_size,
        n_features=len(feature_columns),
        n_informative=4,
        n_redundant=0,
        random_state=None,
        class_sep=0.9,
        flip_y=0.08,
    )
    current_df = pd.DataFrame(X_current, columns=feature_columns)
    return current_df, y_current

def update_drift_metrics(current_df=None, y_current=None):
    if current_df is None or y_current is None:
        current_df, y_current = build_current_batch(sample_size=200)

    reference_sample = reference_data.sample(
        n=min(len(reference_data), len(current_df)),
        random_state=42,
    )
    is_data_drift, drift_scores, overall_drift_score = detect_data_drift(reference_sample, current_df)
    data_drift_gauge.set(float(overall_drift_score))

    X_reference = reference_sample[feature_columns]
    y_reference = model.predict(X_reference)
    is_concept_drift, concept_drift_score = detect_concept_drift(
        model,
        X_reference,
        y_reference,
        current_df[feature_columns],
        y_current,
    )
    concept_drift_gauge.set(float(concept_drift_score))

    return {
        "is_data_drift": bool(is_data_drift),
        "is_concept_drift": bool(is_concept_drift),
        "data_drift_score": round(float(overall_drift_score), 4),
        "concept_drift_score": round(float(concept_drift_score), 4),
        "feature_drift": {name: round(float(score), 4) for name, score in drift_scores.items()},
    }

@app.get("/")
def home():
    base_url = request.host_url.rstrip("/")
    return jsonify({
        "message": "Grafana/Prometheus tutorial app is running",
        "metrics": f"{base_url}/metrics",
        "predict": f"{base_url}/predict",
        "model_path": str(model_path),
    })

@app.get("/predict")
def predict():
    with prediction_latency.time():
        current_df, _ = build_current_batch(sample_size=1)
        probability = float(model.predict_proba(current_df)[0][1])
        prediction_class = classify_prediction(probability)
        confidence = float(max(probability, 1.0 - probability))
        band = confidence_band(confidence)
        status = "success"

        prediction_counter.labels(prediction_class=prediction_class, status=status).inc()
        prediction_confidence_counter.labels(confidence_band=band).inc()
        last_prediction.set(probability)
        last_prediction_confidence.set(confidence)

        drift_summary = update_drift_metrics()

        return jsonify({
            "prediction_score": round(probability, 4),
            "prediction_class": prediction_class,
            "confidence": round(confidence, 4),
            "confidence_band": band,
            "status": status,
            "data_drift_score": drift_summary["data_drift_score"],
            "concept_drift_score": drift_summary["concept_drift_score"],
            "is_data_drift": drift_summary["is_data_drift"],
            "is_concept_drift": drift_summary["is_concept_drift"],
        })

@app.get("/metrics")
def metrics():
    return generate_latest(tutorial_registry), 200, {"Content-Type": "text/plain; version=0.0.4; charset=utf-8"}

class ServerThread(Thread):
    def __init__(self, flask_app, host="0.0.0.0", port=8000):
        super().__init__(daemon=True)
        self.server = make_server(host, port, flask_app)
        self.context = flask_app.app_context()
        self.context.push()

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()

existing_server = globals().get("server_thread")
if existing_server is not None:
    existing_server.shutdown()

existing_scheduler = globals().get("drift_scheduler")
if existing_scheduler is not None:
    existing_scheduler.shutdown(wait=False)

drift_scheduler = BackgroundScheduler()
drift_scheduler.add_job(update_drift_metrics, "interval", seconds=15, id="drift_refresh", replace_existing=True)
drift_scheduler.start()
initial_drift_summary = update_drift_metrics()

server_thread = ServerThread(app)
server_thread.start()

app_script = '''from pathlib import Path
from threading import Thread
from time import sleep
import sys

import joblib
import pandas as pd
from apscheduler.schedulers.background import BackgroundScheduler
from flask import Flask, jsonify, request
from prometheus_client import CollectorRegistry, Counter, Gauge, Histogram, generate_latest
from sklearn.datasets import make_classification
from werkzeug.serving import make_server

project_dir = Path(__file__).resolve().parent.parent
src_dir = project_dir / "src"
models_dir = project_dir / "models"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from concept_drift import detect_concept_drift
from data_drift import detect_data_drift

model_path = models_dir / "tutorial_model.joblib"
reference_data_path = models_dir / "reference_data.joblib"
model = joblib.load(model_path)
reference_data = joblib.load(reference_data_path)
feature_columns = list(reference_data.columns)

app = Flask("grafana_prometheus_tutorial")
tutorial_registry = CollectorRegistry()

prediction_counter = Counter(
    "tutorial_predictions_total",
    "Total number of tutorial prediction requests",
    labelnames=["prediction_class", "status"],
    registry=tutorial_registry,
)
prediction_latency = Histogram(
    "tutorial_prediction_latency_seconds",
    "Latency of tutorial prediction requests",
    registry=tutorial_registry,
)
prediction_confidence_counter = Counter(
    "tutorial_prediction_confidence_total",
    "Predictions grouped by confidence band",
    labelnames=["confidence_band"],
    registry=tutorial_registry,
)
last_prediction = Gauge(
    "tutorial_last_prediction",
    "Latest prediction value returned by the tutorial app",
    registry=tutorial_registry,
)
last_prediction_confidence = Gauge(
    "tutorial_last_prediction_confidence",
    "Latest prediction confidence score returned by the tutorial app",
    registry=tutorial_registry,
)
data_drift_gauge = Gauge(
    "tutorial_data_drift_score",
    "Average feature drift score against the reference dataset",
    registry=tutorial_registry,
)
concept_drift_gauge = Gauge(
    "tutorial_concept_drift_score",
    "Relative performance drop between reference and current batches",
    registry=tutorial_registry,
)

def classify_prediction(score):
    return "positive" if score >= 0.5 else "negative"

def confidence_band(confidence):
    if confidence < 0.4:
        return "low"
    if confidence < 0.75:
        return "medium"
    return "high"

def build_current_batch(sample_size=200):
    X_current, y_current = make_classification(
        n_samples=sample_size,
        n_features=len(feature_columns),
        n_informative=4,
        n_redundant=0,
        random_state=None,
        class_sep=0.9,
        flip_y=0.08,
    )
    current_df = pd.DataFrame(X_current, columns=feature_columns)
    return current_df, y_current

def update_drift_metrics(current_df=None, y_current=None):
    if current_df is None or y_current is None:
        current_df, y_current = build_current_batch(sample_size=200)

    reference_sample = reference_data.sample(
        n=min(len(reference_data), len(current_df)),
        random_state=42,
    )
    is_data_drift, drift_scores, overall_drift_score = detect_data_drift(reference_sample, current_df)
    data_drift_gauge.set(float(overall_drift_score))

    X_reference = reference_sample[feature_columns]
    y_reference = model.predict(X_reference)
    is_concept_drift, concept_drift_score = detect_concept_drift(
        model,
        X_reference,
        y_reference,
        current_df[feature_columns],
        y_current,
    )
    concept_drift_gauge.set(float(concept_drift_score))

    return {
        "is_data_drift": bool(is_data_drift),
        "is_concept_drift": bool(is_concept_drift),
        "data_drift_score": round(float(overall_drift_score), 4),
        "concept_drift_score": round(float(concept_drift_score), 4),
        "feature_drift": {name: round(float(score), 4) for name, score in drift_scores.items()},
    }

@app.get("/")
def home():
    base_url = request.host_url.rstrip("/")
    return jsonify({
        "message": "Grafana/Prometheus tutorial app is running",
        "metrics": f"{base_url}/metrics",
        "predict": f"{base_url}/predict",
        "model_path": str(model_path),
    })

@app.get("/predict")
def predict():
    with prediction_latency.time():
        current_df, _ = build_current_batch(sample_size=1)
        probability = float(model.predict_proba(current_df)[0][1])
        prediction_class = classify_prediction(probability)
        confidence = float(max(probability, 1.0 - probability))
        band = confidence_band(confidence)
        status = "success"

        prediction_counter.labels(prediction_class=prediction_class, status=status).inc()
        prediction_confidence_counter.labels(confidence_band=band).inc()
        last_prediction.set(probability)
        last_prediction_confidence.set(confidence)

        drift_summary = update_drift_metrics()

        return jsonify({
            "prediction_score": round(probability, 4),
            "prediction_class": prediction_class,
            "confidence": round(confidence, 4),
            "confidence_band": band,
            "status": status,
            "data_drift_score": drift_summary["data_drift_score"],
            "concept_drift_score": drift_summary["concept_drift_score"],
            "is_data_drift": drift_summary["is_data_drift"],
            "is_concept_drift": drift_summary["is_concept_drift"],
        })

@app.get("/metrics")
def metrics():
    return generate_latest(tutorial_registry), 200, {"Content-Type": "text/plain; version=0.0.4; charset=utf-8"}

class ServerThread(Thread):
    def __init__(self, flask_app, host="0.0.0.0", port=8000):
        super().__init__(daemon=True)
        self.server = make_server(host, port, flask_app)
        self.context = flask_app.app_context()
        self.context.push()

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()

if __name__ == "__main__":
    scheduler = BackgroundScheduler()
    scheduler.add_job(update_drift_metrics, "interval", seconds=15, id="drift_refresh", replace_existing=True)
    scheduler.start()
    update_drift_metrics()

    server_thread = ServerThread(app)
    server_thread.start()
    print("Tutorial app started.")
    print("Open http://localhost:8000/")
    print("Open http://localhost:8000/metrics")
    print("Call http://localhost:8000/predict a few times to generate metrics.")

    try:
        while True:
            sleep(1)
    except KeyboardInterrupt:
        scheduler.shutdown(wait=False)
        server_thread.shutdown()
'''

app_script_path = src_dir / "app.py"
app_script_path.write_text(app_script, encoding="utf-8")

print("Tutorial app started in the notebook kernel.")
print("Open http://localhost:8000/")
print("Open http://localhost:8000/metrics")
print("Call http://localhost:8000/predict a few times to generate metrics.")
print(f"Wrote reusable app script to {app_script_path}")
print(f"Initial drift summary: {initial_drift_summary}")

Wrote runnable app script to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src\app.py
Tutorial app started.
Open http://127.0.0.1:8000/
Open http://127.0.0.1:8000/metrics
Call http://127.0.0.1:8000/predict a few times to generate metrics.
Loaded model from d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\models\tutorial_model.joblib
Wrote runnable app script to d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\src\app.py
Background drift refresh scheduled every 15 seconds.
Initial drift summary: {'is_data_drift': True, 'is_concept_drift': True, 'data_drift_score': 0.3414, 'concept_drift_score': 0.63, 'feature_drift': {'feature_0': 0.0177, 'feature_1': 0.2182, 'feature_2': 0.0747, 'feature_3': 0.6644, 'feature_4': 0.5403, 'feature_5': 0.5331}}


## Step 4: Generate sample model traffic

Run the next cell to hit the `/predict` endpoint several times.

This now produces a mix of:

- positive and negative classes
- medium and high confidence bands in most runs
- data drift scores
- concept drift scores

Those responses give you both tabular notebook output and Prometheus time series to inspect in later steps.

In [33]:
import requests
import pandas as pd

responses = []
for _ in range(25):
    response = requests.get("http://127.0.0.1:8000/predict", timeout=5)
    response.raise_for_status()
    responses.append(response.json())

traffic_df = pd.DataFrame(responses)
traffic_df.head()

127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -


127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:24] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:25] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:26] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:26] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:26] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:27] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:27] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:27] "GET /predict HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:27] "GET /predict HTTP/1.1" 2

,concept_drift_score,confidence,confidence_band,data_drift_score,is_concept_drift,is_data_drift,prediction_class,prediction_score,status
0,0.395,0.7289,medium,0.3698,True,True,negative,0.2711,success
1,0.605,0.7208,medium,0.2058,True,True,negative,0.2792,success
2,0.395,0.7437,medium,0.2096,True,True,positive,0.7437,success
3,0.510,0.9128,high,0.1240,True,True,positive,0.9128,success
4,0.590,0.6058,medium,0.1908,True,True,positive,0.6058,success


## Step 5: Configure Prometheus to scrape the notebook app

Prometheus is already running in Docker on your machine, so the next step is to point it at the Flask app running on the Windows host.

This tutorial uses `host.docker.internal:8000` because Prometheus runs in a container and the Flask app runs from the notebook kernel on the host machine.

A ready-to-use config file has been created in this project as `prometheus.yml`.

In [7]:
from pathlib import Path

prometheus_config_path = Path.cwd() / "prometheus.yml"
print(prometheus_config_path)
print()
print(prometheus_config_path.read_text())

d:\OneDrive - Hogeschool Rotterdam\1_CURRENT_DOCUMENTS\DATALAB_ALIGNMENT\PROMETHEUS+GRAFANA\grafana_model_monitoring\prometheus.yml

global:
  scrape_interval: 5s
  evaluation_interval: 5s

scrape_configs:
  - job_name: tutorial_flask_app
    static_configs:
      - targets:
          - host.docker.internal:8000



## Step 6: Start or restart Prometheus with the project config

If your Prometheus container is already running with a different config, stop and recreate it with the command below from PowerShell in this project folder.

```powershell
docker rm -f prometheus
docker run -d --name prometheus -p 9090:9090 -v "${PWD}\prometheus.yml:/etc/prometheus/prometheus.yml" prom/prometheus
```

After that, open `http://localhost:9090/targets` and confirm the `tutorial_flask_app` target is `UP`.

## Step 7: Verify the metrics endpoint from the notebook

This cell checks that the metrics endpoint is reachable and that both the prediction metrics and the drift metrics are present before you switch to Prometheus and Grafana.

Because the app now refreshes drift metrics every 15 seconds in the background, you may see the drift gauges change even before you rerun the traffic cell. The same runnable app code is also written to `src/app.py`.

## Step 8: Run the generated app outside Jupyter

> The notebook writes the same Flask monitoring service to `src/app.py`, so you can run it directly from PowerShell without keeping the notebook kernel alive.

Use these commands from the project folder:

```powershell
conda activate grafana_tutorial
python src/app.py
```

Expected result:

- App endpoint: `http://127.0.0.1:8000/`
- Metrics endpoint: `http://127.0.0.1:8000/metrics`
- Drift gauges refresh every 15 seconds in the background via APScheduler

If you already started the app from the notebook, stop that notebook-hosted server first or restart the kernel before launching `src/app.py` so port `8000` is free.

In [34]:
metrics_response = requests.get("http://127.0.0.1:8000/metrics", timeout=5)
metrics_response.raise_for_status()

metrics_text = metrics_response.text
expected_metric_names = [
    "tutorial_predictions_total",
    "tutorial_prediction_confidence_total",
    "tutorial_last_prediction",
    "tutorial_last_prediction_confidence",
    "tutorial_data_drift_score",
    "tutorial_concept_drift_score",
]

for metric_name in expected_metric_names:
    print(metric_name, metric_name in metrics_text)

print()
print("\n".join(line for line in metrics_text.splitlines() if "tutorial_" in line))

127.0.0.1 - - [28/Jun/2026 11:25:28] "GET /metrics HTTP/1.1" 200 -


tutorial_predictions_total True
tutorial_prediction_confidence_total True
tutorial_last_prediction True
tutorial_last_prediction_confidence True
tutorial_data_drift_score True
tutorial_concept_drift_score True

# HELP tutorial_predictions_total Total number of tutorial prediction requests
# TYPE tutorial_predictions_total counter
tutorial_predictions_total{prediction_class="negative",status="success"} 9.0
tutorial_predictions_total{prediction_class="positive",status="success"} 16.0
# HELP tutorial_predictions_created Total number of tutorial prediction requests
# TYPE tutorial_predictions_created gauge
tutorial_predictions_created{prediction_class="negative",status="success"} 1.782638724306106e+09
tutorial_predictions_created{prediction_class="positive",status="success"} 1.7826387243473146e+09
# HELP tutorial_prediction_latency_seconds Latency of tutorial prediction requests
# TYPE tutorial_prediction_latency_seconds histogram
tutorial_prediction_latency_seconds_bucket{le="0.005"} 0.0


127.0.0.1 - - [28/Jun/2026 11:25:29] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:34] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:39] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:44] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:49] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:54] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:25:59] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:04] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:09] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:14] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:19] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:24] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:26:29] "GET /metrics HTTP/1.1" 200 -


## Step 9: PromQL queries for model monitoring and drift

> Once the target is `UP` in `http://localhost:9090/targets`, open the Prometheus expression browser and try these queries.

> These queries cover traffic, confidence, latency, and the two drift gauges exposed by the notebook app.

In [30]:
promql_examples = {
    "total_predictions": "sum(tutorial_predictions_total)",
    "predictions_by_class": "sum by (prediction_class) (tutorial_predictions_total)",
    "predictions_per_minute": "sum(rate(tutorial_predictions_total[1m])) * 60",
    "predictions_per_minute_by_class": "sum by (prediction_class) (rate(tutorial_predictions_total[1m])) * 60",
    "confidence_band_distribution": "sum by (confidence_band) (tutorial_prediction_confidence_total)",
    "p95_latency_seconds": "histogram_quantile(0.95, sum(rate(tutorial_prediction_latency_seconds_bucket[5m])) by (le))",
    "average_latency_seconds": "rate(tutorial_prediction_latency_seconds_sum[5m]) / rate(tutorial_prediction_latency_seconds_count[5m])",
    "latest_prediction_score": "tutorial_last_prediction",
    "latest_prediction_confidence": "tutorial_last_prediction_confidence",
    "current_data_drift_score": "tutorial_data_drift_score",
    "current_concept_drift_score": "tutorial_concept_drift_score",
}

for name, query in promql_examples.items():
    print(f"{name}:\n  {query}\n")

total_predictions:
  sum(tutorial_predictions_total)

predictions_by_class:
  sum by (prediction_class) (tutorial_predictions_total)

predictions_per_minute:
  sum(rate(tutorial_predictions_total[1m])) * 60

predictions_per_minute_by_class:
  sum by (prediction_class) (rate(tutorial_predictions_total[1m])) * 60

confidence_band_distribution:
  sum by (confidence_band) (tutorial_prediction_confidence_total)

p95_latency_seconds:
  histogram_quantile(0.95, sum(rate(tutorial_prediction_latency_seconds_bucket[5m])) by (le))

average_latency_seconds:
  rate(tutorial_prediction_latency_seconds_sum[5m]) / rate(tutorial_prediction_latency_seconds_count[5m])

latest_prediction_score:
  tutorial_last_prediction

latest_prediction_confidence:
  tutorial_last_prediction_confidence

current_data_drift_score:
  tutorial_data_drift_score

current_concept_drift_score:
  tutorial_concept_drift_score



127.0.0.1 - - [28/Jun/2026 11:23:59] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:04] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:09] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:14] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:19] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:24] "GET /metrics HTTP/1.1" 200 -
127.0.0.1 - - [28/Jun/2026 11:24:29] "GET /metrics HTTP/1.1" 200 -


## Step 10: Containerize the tutorial stack on Windows

> If you want a setup closer to the original article, this project now includes real Docker artifacts in the project root so you do not need to copy the examples by hand.

Files now available:

- `Dockerfile`
- `docker-compose.yml`
- `prometheus.docker.yml`

The notebook-first flow still works for development. The Docker stack is the cleaner option when you want a repeatable demo with the app, Prometheus, and Grafana all starting together.

The Docker Compose setup uses `prometheus.docker.yml`, which scrapes the app by container name `tutorial-app:8000`. That keeps the existing `prometheus.yml` unchanged for the notebook-hosted Windows flow.

Start the stack from PowerShell in the project folder:

```powershell
docker compose up --build -d
```

After startup:

- open Grafana at `http://localhost:3000`
- open Prometheus at `http://localhost:9090/targets`
- confirm the `tutorial_flask_app` target is `UP`

To stop the stack:

```powershell
docker compose down
```

## Step 11: Choose the right Prometheus config

> This project now has two Prometheus config files because the scrape target changes depending on where the Flask app is running.

Use `prometheus.yml` when:

- the Flask app is running from the notebook kernel or from PowerShell on Windows
- Prometheus is running in Docker and needs to reach the Windows host through `host.docker.internal:8000`

Use `prometheus.docker.yml` when:

- the Flask app is running inside Docker Compose as the `tutorial-app` service
- Prometheus should scrape the app over the Docker network using `tutorial-app:8000`

In short:

- notebook or PowerShell app on Windows -> `prometheus.yml`
- full Docker Compose stack -> `prometheus.docker.yml`

## Step 12: Add Prometheus as a Grafana data source

> Open Grafana at `http://localhost:3000/` and add Prometheus as a data source.

Use these settings:

- Name: `Prometheus`
- URL: `http://host.docker.internal:9090` if Grafana runs in Docker and the app runs on Windows
- URL: `http://localhost:9090` if Grafana runs directly on Windows
- URL: `http://prometheus:9090` if you use the Docker Compose stack from the previous steps
- Access: default is fine for a local tutorial

Click **Save & test**. The result should say that the data source is working.

## Step 13: Build the first model-monitoring Grafana dashboard

> Create a new dashboard and add these panels.

### Panel 1: Total predictions
- Visualization: `Stat`
- Query: `sum(tutorial_predictions_total)`

### Panel 2: Predictions by class
- Visualization: `Pie chart` or `Bar chart`
- Query: `sum by (prediction_class) (tutorial_predictions_total)`

### Panel 3: Predictions per minute by class
- Visualization: `Time series`
- Query: `sum by (prediction_class) (rate(tutorial_predictions_total[1m])) * 60`

### Panel 4: Confidence band distribution
- Visualization: `Bar chart`
- Query: `sum by (confidence_band) (tutorial_prediction_confidence_total)`

### Panel 5: P95 latency
- Visualization: `Time series`
- Query: `histogram_quantile(0.95, sum(rate(tutorial_prediction_latency_seconds_bucket[5m])) by (le))`

### Panel 6: Latest prediction confidence
- Visualization: `Gauge`
- Query: `tutorial_last_prediction_confidence`

### Panel 7: Data drift score
- Visualization: `Gauge` or `Time series`
- Query: `tutorial_data_drift_score`

### Panel 8: Concept drift score
- Visualization: `Gauge` or `Time series`
- Query: `tutorial_concept_drift_score`

A good dashboard title for this notebook is `Tutorial Model Monitoring Dashboard`.

## Step 14: Generate more traffic while Grafana is open

> If the dashboard looks flat, rerun the traffic cell from Step 4 a few times. That will make the rate, latency, and drift panels visibly change.

You can also increase the loop count in the request cell from `25` to `50` for a more obvious signal.